[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [22]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [23]:
import torch
import torch.nn as nn

In [44]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    acc_loss = 0
    optimizer.zero_grad()
    for x, y in micro_batches:
      loss = loss_fn(model(x), y) / len(micro_batches)
      loss.backward()
      acc_loss += loss.item()
    optimizer.step()
    return acc_loss

In [45]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 1.4375793486833572


In [46]:
# ✅ SUBMIT
from torch_judge import check, hint
check('gradient_accumulation')
hint("gradient_accumulation")


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (4.6ms)
  ✅ [2/3] Returns loss value (0.6ms)
  ✅ [3/3] Parameters actually update (0.6ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (5.8ms total)
  Progress saved. Run status() to see your dashboard.


💡 Hint for Gradient Accumulation:
   Zero grads once. For each micro-batch: forward, loss/n_batches, backward. Then optimizer.step(). The loss scaling ensures accumulated grads match a single large batch.

